# Scenario comparison

Compares month-end **nominal** net worth across any number of scenarios against a
baseline, and finds each variant's break-even month — the point after which it
stays ahead for good.

Pick a baseline and up to 4 variant scenarios below — every chart and table in
this notebook updates automatically as the selection changes, no re-running
required. Each must already have a run (`analysis sim load <name> && analysis
sim run <name>`); the newest run for each is used.

> **Note:** after running cells, `git status` will show this notebook as modified
> even with no real changes — see [notebooks/README.md](./README.md#git-status-shows-a-notebook-as-modified-after-running-it).

> **Table clipped, or the notebook slow?** Run `analysis jupyter` and open this notebook in JupyterLab: wide tables scroll sideways there, and it can use less memory than VS Code.

In [ ]:
%load_ext autoreload
%autoreload 2

from analysis.sim.notebook_ui.pickers import (
    DollarModePicker, TimeRangePicker, ValuePlotModePicker, multi_scenario_picker,
)
from analysis.sim.notebook_ui.compare_plots import (
    comparability_widget, rules_and_events_widget,
    net_worth_compare_widget, breakeven_widget, year_end_widget,
    show_account_balance_diffs, guard_rails_compare_widget, strategy_compare_widget,
    spending_compare_widget,
)
from analysis.sim.notebook_ui.lot_compare import account_definitions_widget
MAX_VARIANTS = 4

# CONNECTIONS holds the live RunConnections, refreshed in place whenever either picker changes.
PICKER, CONNECTIONS = multi_scenario_picker(default_baseline="base", max_variants=MAX_VARIANTS)
DOLLAR_MODE = DollarModePicker()
VALUE_MODE = ValuePlotModePicker()
# The years the charts and tables below show; it starts at the whole span of the runs.
TIME_RANGE = TimeRangePicker()
PICKER

## Are these runs comparable?

A comparison is only meaningful if both runs came from the same code and the same
data version — otherwise a difference here could be an unrelated edit rather than
the scenario change under test.


In [ ]:
CONNECTIONS.show(comparability_widget)

## What actually differs?

Each scenario inherits everything from its parent except the rows it overrides, so
this is the whole mechanism behind any difference in the charts below. If a rule
you expected to change isn't listed, the override didn't take effect.

Two sources of difference are shown separately:

- **Rules** — as they stand at the *end* of the run. A calendar event that edits a
  rule mid-run (`set_amount`) shows up here as a changed amount, because the run
  snapshot records the post-event state.
- **Calendar events** — one-time interventions, including the benefit cut.


In [ ]:
CONNECTIONS.show(rules_and_events_widget)

### Accounts, lots and year-to-date rows

Accounts and lots that are *defined* differently: an account's own settings, then one
sub-row per lot (`symbol`, `acquired`, `quantity`, `series`, and the opening balance,
cost basis and prior-year-end balance, including balances derived from a quantity and a
price). A lot only some scenarios have shows `(exists)` as `—` in the others.

Year-to-date rows are normally inherited, so they get their own table: only rows that
differ are shown.

In [ ]:
CONNECTIONS.show(account_definitions_widget)

## Net worth over time

Use the shared **Dollar basis** toggle to switch between nominal and CPI-constant dollars. The selected basis applies to the net-worth chart and the comparison tables below.

In [ ]:
CONNECTIONS.show(net_worth_compare_widget, dollar_mode=DOLLAR_MODE, time_range=TIME_RANGE)

### Break-even

**Break-even** is the first month after which the variant stays ahead for the rest
of the scenario — not merely the first month it pokes above zero, which can happen
transiently when a lumpy payment lands in one scenario before the other.


In [ ]:
CONNECTIONS.show(breakeven_widget, dollar_mode=DOLLAR_MODE)

## Account Comparisons

Pick an account to compare its balance. For an account with more than one lot, a
**Lot** dropdown appears under it; once a lot is picked, **Measure** shows that lot's
**balance**, **cost basis** or **quantity** (basis and quantity often explain a difference
in balance or in gains).

In [ ]:
CONNECTIONS.show(
    show_account_balance_diffs,
    dollar_mode=DOLLAR_MODE,
    value_mode=VALUE_MODE,
    time_range=TIME_RANGE,
)

### Annual net worth comparisons and deltas.|

In [ ]:
CONNECTIONS.show(year_end_widget, dollar_mode=DOLLAR_MODE, time_range=TIME_RANGE)

## Guard rails

Each scenario's margins against its guard rails (`guard_rails.csv`; see the [guard rails guide](../docs/guides/guard-rails.md)), side by side; `sim compare --section guard-rails` prints the same tables. Rails are matched by name, so a rail only some scenarios define shows only for those.

- A **margin** is the room left before a rail: negative is past it, whichever way the rail faces. The **annual** margin is one year's; the **cumulative** margin sums the years so far, so it says whether a year is using room banked earlier.
- **Chart**: a row per rail, annual on the left and cumulative on the right, a line per scenario (the baseline solid, the variants dashed). Below zero is past the rail, tinted warm for a rail to stay inside and cool, hatched, for a positive one (where past it means room to spend more).
- **Tables**: per rail, each scenario's margin by year, and each variant's `Δ` (variant less baseline, so a positive Δ is more room); cells past a rail are red, or blue for a positive rail. Then each scenario's whole-run summary: years past, the worst margin and its year, the margin at the end, and the RMS overshoot.
- The dollars are the shared **Dollar basis**; the years picked only limit what is shown, since margins accumulate from each rail's start. Rails are read from the data repo at each draw, so an edited rail shows without a re-run.

In [ ]:
CONNECTIONS.show(guard_rails_compare_widget, dollar_mode=DOLLAR_MODE, time_range=TIME_RANGE)

## Strategies: on vs off

Each run's allocation drift against the same target set (a strategy's `targets` and `portfolio` in any of the runs, the baseline's first): a panel per asset class and one of the largest drift either way, a line per scenario, with the strategy's band shaded. Then what each scenario's strategies did over the run: the dollars asked of the chains they own, how much the strategy moved itself, and how much it left to the chain's steps.

To see what a strategy changes, compare its scenario with a child that switches it off (`sim strategy edit NAME --disable --scenario CHILD` writes the override). The net worth and break-even sections above then show what it costs or gains. `sim compare --section strategies` prints the same tables.

In [ ]:
CONNECTIONS.show(strategy_compare_widget, time_range=TIME_RANGE)

## Spending against the plan

Each run's delivered spending by year, a line per scenario, with its plan dashed in the same colour where a spending strategy moved it off the plan; then delivered spending by year with each variant's difference, and each run's whole span against its plan (the lowest year is the lowest whole year, since a part year at either end would always be lowest). `sim compare --section spending` prints the same tables.

In [ ]:
CONNECTIONS.show(spending_compare_widget, dollar_mode=DOLLAR_MODE, time_range=TIME_RANGE)